In [ ]:
# Repository bootstrap — portable local/Jupyter execution
from pathlib import Path
import os, sys

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "koa_severity").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from koa_severity.data import get_dataset_path
DATASET_PATH = get_dataset_path()
OUTPUTS_DIR = PROJECT_ROOT / "results"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
OUTPUTS_DIR.mkdir(exist_ok=True)
CHECKPOINTS_DIR.mkdir(exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)


In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import roc_auc_score
import os
import timm

# ۱. دریافت آدرس دیتاسِت از kagglehub
dataset_path = get_dataset_path()

test_dir = None
for root, dirs, files in os.walk(dataset_path):
    for d in dirs:
        if d.lower() == 'test':
            test_dir = os.path.join(root, d)
            break

print(f"🔍 Test folder found at: {test_dir}")

# ۲. یافتن خودکار فایل .pth آپلودشده در محیط اصلی کولب
files_in_colab = [f for f in os.listdir('.') if f.endswith('.pth') or f.endswith('.pt')]
model_path = files_in_colab[0]
print(f"🎯 Loading model from: '{model_path}'")

# ۳. تنظیمات Device و Preprocessing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

test_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ۴. لود کردن مدل ConvNeXt (اضافه شدن weights_only=False برای رفع ارور Unpickling)
model = timm.create_model('convnext_small.fb_in22k_ft_in1k_384', pretrained=False, num_classes=5)

checkpoint = torch.load(model_path, map_location=device, weights_only=False)

state_dict = checkpoint.get('model_state_dict', checkpoint.get('state_dict', checkpoint))

new_state_dict = {}
for key, value in state_dict.items():
    new_key = key.replace("backbone.", "", 1) if key.startswith("backbone.") else key
    new_state_dict[new_key] = value

model.load_state_dict(new_state_dict, strict=False)
model.to(device)
model.eval()

# ۵. محاسبه پیش‌بینی‌ها
all_probs = []
all_labels = []

print("⏳ Running inference for Focal Loss Model...")
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs = np.vstack(all_probs)
all_labels = np.concatenate(all_labels)

# ۶. محاسبه Macro ROC-AUC
y_true_onehot = np.eye(5)[all_labels]
macro_auc = roc_auc_score(y_true_onehot, all_probs, average='macro', multi_class='ovr')

print("\n" + "="*45)
print(f"🎯 Focal Loss Model Macro ROC-AUC: {macro_auc:.4f}")
print("="*45)

class_names = ['KL 0 (Normal)', 'KL 1 (Doubtful)', 'KL 2 (Mild)', 'KL 3 (Moderate)', 'KL 4 (Severe)']
for i, name in enumerate(class_names):
    auc = roc_auc_score(y_true_onehot[:, i], all_probs[:, i])
    print(f"• {name:18s}: {auc:.4f}")

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import roc_auc_score
import os
import timm

# ۱. دانلود دیتاسِت
dataset_path = get_dataset_path()

test_dir = None
for root, dirs, files in os.walk(dataset_path):
    for d in dirs:
        if d.lower() == 'test':
            test_dir = os.path.join(root, d)
            break

# ۲. دانلود مستقیم فایل وزن از گوگل درایو (بدون نیاز به آپلود دستی)
# شناسه ID فایل شما از درایو: 16GZpzf-mtYgN7SzNrmtjDOBA35tfeyxW
model_path = 'convnext_s_384_focalloss.pth'

if not os.path.exists(model_path):
    print("⏳ Downloading checkpoint file directly via gdown...")
    os.system(f'gdown --id 16GZpzf-mtYgN7SzNrmtjDOBA35tfeyxW -O {model_path}')

# ۳. تنظیمات GPU و Preprocessing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

test_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ۴. بارگذاری مدل
model = timm.create_model('convnext_small.fb_in22k_ft_in1k_384', pretrained=False, num_classes=5)

checkpoint = torch.load(model_path, map_location=device, weights_only=False)

state_dict = checkpoint.get('model_state_dict', checkpoint.get('state_dict', checkpoint))

new_state_dict = {}
for key, value in state_dict.items():
    new_key = key.replace("backbone.", "", 1) if key.startswith("backbone.") else key
    new_state_dict[new_key] = value

model.load_state_dict(new_state_dict, strict=False)
model.to(device)
model.eval()

# ۵. محاسبه پیش‌بینی‌ها
all_probs = []
all_labels = []

print("⏳ Running inference on GPU...")
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs = np.vstack(all_probs)
all_labels = np.concatenate(all_labels)

# ۶. خروجی ROC-AUC
y_true_onehot = np.eye(5)[all_labels]
macro_auc = roc_auc_score(y_true_onehot, all_probs, average='macro', multi_class='ovr')

print("\n" + "="*45)
print(f"🎯 Focal Loss Model Macro ROC-AUC: {macro_auc:.4f}")
print("="*45)

class_names = ['KL 0 (Normal)', 'KL 1 (Doubtful)', 'KL 2 (Mild)', 'KL 3 (Moderate)', 'KL 4 (Severe)']
for i, name in enumerate(class_names):
    auc = roc_auc_score(y_true_onehot[:, i], all_probs[:, i])
    print(f"• {name:18s}: {auc:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import roc_auc_score, classification_report
import os
import timm

# ۱. دریافت دیتاسِت از Kaggle
dataset_path = get_dataset_path()

def find_folder(name):
    for root, dirs, files in os.walk(dataset_path):
        for d in dirs:
            if d.lower() == name.lower():
                return os.path.join(root, d)
    return None

train_dir = find_folder('train')
val_dir = find_folder('val') or find_folder('validation')
test_dir = find_folder('test')

print(f"📁 Train: {train_dir}\n📁 Val: {val_dir}\n📁 Test: {test_dir}")

# ۲. تنظیمات Device و Data Augmentation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Running on: {device}")

train_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=eval_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ۳. تعریف Weighted Focal Loss
class WeightedFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super(WeightedFocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# محاسبه وزن کلاس‌ها برای مدیریت نابرابری داده‌ها
class_counts = [0] * 5
for _, label in train_dataset.samples:
    class_counts[label] += 1

total_samples = sum(class_counts)
class_weights = [total_samples / (5.0 * count) for count in class_counts]
class_weights = torch.FloatTensor(class_weights).to(device)

criterion = WeightedFocalLoss(gamma=2.0, alpha=class_weights)

# ۴. تعریف مدل ConvNeXt
model = timm.create_model('convnext_small.fb_in22k_ft_in1k_384', pretrained=True, num_classes=5)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# ۵. حلقه آموزش (Training Loop)
best_val_f1 = 0.0
save_path = 'best_focal_convnext_retrained.pth'
epochs = 15

print("\n🏋️ Starting Model Retraining...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    # ارزیابی روی Validation Set
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_report = classification_report(val_targets, val_preds, output_dict=True, zero_division=0)
    val_f1 = val_report['macro avg']['f1-score']

    print(f"Epoch [{epoch+1:02d}/{epochs:02d}] - Loss: {running_loss/len(train_dataset):.4f} - Val Macro F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({'state_dict': model.state_dict()}, save_path)
        print(f"  └─ 🎯 New Best Model Saved! (Val F1: {val_f1:.4f})")

# ۶. ارزیابی نهایی و محاسبه دقیق Macro ROC-AUC روی Test Set
print("\n" + "="*50)
print("⏳ Running Final Evaluation on Test Set...")
print("="*50)

checkpoint = torch.load(save_path, map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

all_probs, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs = np.vstack(all_probs)
all_labels = np.concatenate(all_labels)

# محاسبه Macro ROC-AUC
y_true_onehot = np.eye(5)[all_labels]
macro_auc = roc_auc_score(y_true_onehot, all_probs, average='macro', multi_class='ovr')

print("\n" + "="*50)
print(f"🔥 FINAL Focal Loss Model Macro ROC-AUC: {macro_auc:.4f}")
print("="*50)

class_names = ['KL 0 (Normal)', 'KL 1 (Doubtful)', 'KL 2 (Mild)', 'KL 3 (Moderate)', 'KL 4 (Severe)']
for i, name in enumerate(class_names):
    auc = roc_auc_score(y_true_onehot[:, i], all_probs[:, i])
    print(f"• {name:18s}: {auc:.4f}")